<a href="https://colab.research.google.com/github/Valementajat/PageRank_colab/blob/master/pageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
''' pip install kaggle '''

' pip install kaggle '

In [2]:
# install PySpark
!apt-get update -y # Update apt-get repository.
!apt-get install -y openjdk-11-jdk-headless  # Install Java.
!wget  https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz # Download Apache Sparks.
!tar xf spark-3.5.1-bin-hadoop3.tgz # Unzip the tgz file.
!pip install  findspark # Install findspark. Adds PySpark to the System path during runtime.


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,934 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,829 kB]
Get:14

In [3]:
import os
os.environ['KAGGLE_USERNAME'] = "XXXXX"
os.environ['KAGGLE_KEY'] = "XXXXX"

# Set environment variables for spark
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

dataset_zip = "new-york-times-articles-comments-2020.zip"
if not os.path.exists(dataset_zip):
  !kaggle datasets download -d  benjaminawd/new-york-times-articles-comments-2020




Dataset URL: https://www.kaggle.com/datasets/benjaminawd/new-york-times-articles-comments-2020
License(s): CC-BY-NC-SA-4.0
100% 1.95G/1.95G [00:20<00:00, 101MB/s]



In [4]:
# Initialize findspark
import findspark
findspark.init()

# Create a PySpark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better

spark

In [ ]:
# Unzip the file independenty
articles = "nyt-articles-2020.csv"
comments = "nyt-comments-2020.csv"

if not os.path.exists(articles) and not os.path.exists(comments):
  !unzip new-york-times-articles-comments-2020.zip

Archive:  new-york-times-articles-comments-2020.zip
  inflating: nyt-articles-2020.csv   
  inflating: nyt-comments-2020.csv   

In [ ]:
import numpy as np
import pandas as pd





In [ ]:
from pyspark.sql import functions as F
articles_df = spark.read.option("header", True).csv("/content/nyt-articles-2020.csv")


#parsing the comments in Spark crash so, we do a bit of trickery
chunk_size=10000
comments_df = None

# I know you're not supposed to do this, but the parsing of the comments .csv didn't want to comply

for chunk in pd.read_csv("nyt-comments-2020.csv", chunksize=chunk_size,  usecols=["userID", "articleID"]):
  # clean current chunk
  chunk = chunk.dropna(subset=["userID", "articleID"]).drop_duplicates()
  # Create spark dataframe
  spark_chunk = spark.createDataFrame(chunk.to_dict("records"))

  # append into one Spark DataFrame
  if comments_df is None:
      comments_df = spark_chunk
  else:
      comments_df = comments_df.unionByName(spark_chunk)

# final dedup in Spark
comments_df = comments_df.dropDuplicates().persist()



In [ ]:
# Lets initialize a sample freq at the start so it's easy to spot
# Sample_fraq is the sample of articles we're concidering, you can set this between 0.01 and 1.0
sample_frac = 0.1

sampled_articles = (
    articles_df
    .select(F.col("uniqueID").alias("articleID"))
    .distinct()
    .sample(False, sample_frac, seed=42)
    )


In [ ]:


# First map
# I know that with spark I could try the RDD, but I already made the mapReduce implementation
# and I don't have that much time
interactions = comments_df.join(sampled_articles, on="articleID", how="inner")



In [ ]:
#Then reduce

# Desided to flip this otherway around as I'm rewriting this again, so now user [articles]
# As this is possibly faster
articleUsers = (
    interactions
    .groupBy("userID")
    .agg(F.collect_set("articleID").alias("articles"))
)

In [ ]:
# Lets create the article pairs where the common users are >= 2
article_pairs = (
    interactions.alias("a")
    .join(
        interactions.alias("b"),
        on="userID",
        how="inner"
    )
    .where(F.col("a.articleID") < F.col("b.articleID"))   # avoid self-pairs and reversed duplicates
    .groupBy(
        F.col("a.articleID").alias("article1"),
        F.col("b.articleID").alias("article2")
    )
    .agg(
        F.count("*").alias("shared_user_count"),

    )
    .where(F.col("shared_user_count") >= 2)
)


In [ ]:
#Making the connections both ways as we only define a "link" between the pages
adjacency = (
    article_pairs.select(
        F.col("article1").alias("src"),
        F.col("article2").alias("dst")
    )
    .unionByName(
        article_pairs.select(
            F.col("article2").alias("src"),
            F.col("article1").alias("dst")
        )
    )
    .dropDuplicates()
    .cache()
)


In [ ]:
# At this point in the night I realize that I didn't have to redo everything with Spark, but distribute the computation of the matrixes.....

In [ ]:
pages = (
    interactions
    .select(F.col("articleID").alias("id"))
    .distinct()
    .cache()
)

In [ ]:
# degree is practically how many times that page is seen in the connections

outdeg = (
    adjacency
    .groupBy("src")
    .count()
    .withColumnRenamed("count", "outdeg")
    .cache()
)

In [ ]:


#sparse transition matrix
# This is a  dataframe and we need convert it to RDD
connection_matrix = (
    adjacency
    .join(outdeg, on="src", how="left")
    .select(
        F.col("dst").alias("dst"),
        F.col("src").alias("src"),
        (F.lit(1.0) / F.col("outdeg")).alias("weight")
    )
    .rdd
    .map(lambda r: (r.dst, r.src, r.weight))
    .cache()
)
''' connection_matrix = (
    adjacency
    .join(outdeg, on="src", how="left")
    .withColumn("weight", F.lit(1.0) / F.col("outdeg"))
    .select("src", "dst", "weight")
    .cache() # parallelized by this
)
# convert to rdd
connection_matrix = connection_matrix.rdd '''



In [ ]:
# Creating an indexed connection matrix
page_ids = [row["id"] for row in pages.orderBy("id").collect()]
id_to_idx = {pid: i for i, pid in enumerate(page_ids)}
n = len(page_ids)
connection_matrix_indexed = (
    connection_matrix
    .map(lambda t: (id_to_idx[t[0]], id_to_idx[t[1]], t[2]))
    .cache()
)

In [ ]:
# A late night implementation for dangling node


dangling_ids = (
    pages
    .join(outdeg, pages.id == outdeg.src, "left_anti")
    .select("id")
    .rdd
    .map(lambda r: id_to_idx[r["id"]])
    .collect()
)

In [ ]:
def l2distance(v, q):
    '''Computes the Euclidean distance between two vectors.

    Args

    v: list or tuple
    q: list or tuple

    Returns

    d: Euclidean distance between v and q

    Throws exception of v and q do not have the same length.
    '''

    if len(v) != len(q):
        raise ValueError('Cannot compute the distance'
                         ' of two vectors of different size')

    return sum([(q_el - v_el)**2 for v_el, q_el in zip(v, q)])

In [ ]:
sc = spark.sparkContext

n = pages.count()

In [ ]:



page_rank = np.ones(n)/n
old_page_rank = np.ones(n)

# to cut time I've set the iterations to 5
max_iterations = 5
iteration = 0
tolerance=10e-7
damping=0.85

dist = l2distance(old_page_rank, page_rank)

while dist >= tolerance and \
          iteration < max_iterations:
          old_page_rank = page_rank
          page_rank_values = (connection_matrix_indexed
                              .map(lambda t: (t[0], t[2]*page_rank[t[1]]))
                              .reduceByKey(lambda a, b: a+b)
                              .collect()
                          )
          dangling_mass = sum(old_page_rank[i] for i in dangling_ids)

          new_page_rank = np.ones(n) * (1 - damping) / n
          new_page_rank += damping * dangling_mass / n

          for i, c in page_rank_values:
              new_page_rank[i] += damping * c
          page_rank = new_page_rank







          iteration += 1
          dist = l2distance(old_page_rank, page_rank)

          print('{} iterations'.format(iteration))

In [ ]:
#show the top ranks
top_k = 20

top_idx = np.argsort(page_rank)[::-1][:top_k]

for i in top_idx:
    print(page_ids[i], page_rank[i])

Below here is the old implementation

In [ ]:
''' chunk_size = 10000

# Using a set to avoid duplicates
ids = set()


# To guard against the memory we load the csv in chuncks and process it right away
for article_chunk in pd.read_csv("nyt-articles-2020.csv", chunksize=chunk_size):
   for article_id in article_chunk["uniqueID"]:
      ids.add(article_id)

n = len(ids) '''

In [ ]:
''' # Lets create the dataset we want to work from the articles
sample_size = int(n * sample_frac)

# Adding a seed to make the sampling reproducible
random.seed(10)
sample_ids = random.sample(list(ids), sample_size)
n = len(sample_ids)
# Maybe this will be a bit more efficient to work with than a full on URL?
id_to_i = {article_id: i for i, article_id in enumerate(sample_ids)} '''



In [ ]:
''' # Creating an empty adjancency matrix
adjencencyMatrix =  np.zeros((n, n ), dtype=float) '''



In [ ]:
''' # TO avoid blowing up the memory usage, we load the
# comments in chucks
chunk_size = 10000

# First part of the mapReduce, we create the key value pairs
KeyValuePairs = []

for chunk in pd.read_csv("nyt-comments-2020.csv", chunksize=chunk_size):
  # Start the MapReduce by creating key-value pairs
  for index, row in chunk.iterrows():
    if row['articleID'] in id_to_i:
      # Currently creating this like "article -> user", at now it's okay i guess
      KeyValuePairs.append((id_to_i[row["articleID"]], row["userID"]))

del chunk
del chunk_size

gc.collect()
 '''


In [ ]:
''' # Lets group the KeyValuePairs by the key
KeyValueGroup = defaultdict(set)
for article_idx, user_id in KeyValuePairs:
    # but
    KeyValueGroup[article_idx].add(user_id)
 '''

In [ ]:
''' # Then the final reduce so we can use this
connectionPairs = []

items = list(KeyValueGroup.items())

# This is a possibly a problem?
# I've created the groups with article [uids], but it would be more efficient
# with uid [articles] ?

# I'm checking items against other values in the same set and looking if there
# are two unique intersecting user pairs so we can say that there
for a_idx, users_a in items:
    for b_idx, users_b in items:
        if a_idx != b_idx:
            if len(users_a.intersection(users_b)) >= 2:
                connectionPairs.append((a_idx, b_idx)) '''


In [ ]:
''' for i in range(10):
  print(connectionPairs[i]) '''

In [ ]:
''' # degree is practically how many times that page is seen in the connections
degree = Counter()
for i, j in connectionPairs:
    degree[i] += 1


for i, j in connectionPairs:
    adjencencyMatrix[i, j] = 1.0 / degree[i] '''


In [ ]:
''' #We have a problem, dangling nodes, so this is to uniformally
# set the propability to go from a page to every page when calculating the rank

row_sums = adjencencyMatrix.sum(axis=1)
dangling = (row_sums == 0)


# replace each dangling row with uniform distribution
adjencencyMatrix[dangling, :] = 1.0 / n '''

In [ ]:
''' # So we have our stochastic adjencencyMatrix or connection matrix

# Without any dangling pages (or nodes)

# Lets run the power iteration

# This is the formula to "teleport", during the power iteration.
# as in, do we take the next page from the page's links or do we take one from
# all of the pages
# P = βM + (1 - β)1/N

beta = 0.85 # example size is between 0.8 - 0.9
n = len(adjencencyMatrix)

#now this is the pageRank transition matrix or the google matrix
P = beta * adjencencyMatrix + (1 - beta) / n '''

In [ ]:
''' # lets initialize the rank vector r^(0) which has every rank as 1/n
r = np.ones(n) / n '''

In [ ]:
''' # Now we calculat the new ranks using the transition matrix and the old rank
# r^new =  r^old · A
for i in range(100):
  # python's matrix manipulation to multiply matrix with another matrix
    r = r @ P '''

In [ ]:
''' # This is just a sanity check
print(r.sum()) '''

In [ ]:
''' # Top 10 indices by rank (largest first)
top_k = 10
top_idx = np.argsort(r)[-top_k:][::-1]

print("Top ranks:")
for idx in top_idx:
    print(idx, r[idx]) '''

In [ ]:
%whos